<a href="https://colab.research.google.com/github/ykchoudhary110/AI-Orchestration-System/blob/main/AiOrchestration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faiss-cpu sentence-transformers pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.9 MB/s eta 0:00:00


In [2]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()

In [3]:
import pandas as pd

df = pd.read_excel("/content/ai_complex_dataset.xlsx")

df['Prompt'] = df['Prompt'].astype(str)
df['Cleaned'] = df['Prompt'].apply(clean_text)

print("Dataset loaded:", len(df))

FileNotFoundError: [Errno 2] No such file or directory: '/content/ai_complex_dataset.xlsx'

In [4]:
import pandas as pd

df = pd.read_excel("/content/ai_complex_dataset.xlsx")

df['Prompt'] = df['Prompt'].astype(str)
df['Cleaned'] = df['Prompt'].apply(clean_text)

print("Dataset loaded:", len(df))

Dataset loaded: 2000


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df['Cleaned'].tolist())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype('float32')

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("FAISS index ready!")

FAISS index ready!


In [7]:
restricted_keywords = [
    "hack password",
    "bypass security",
    "illegal access",
    "crack software",
    "generate malware",
    "ddos attack",
    "phishing attack"
]

def is_restricted(text):
    return any(word in text for word in restricted_keywords)

In [8]:
restricted_keywords = [
    "hack password",
    "bypass security",
    "illegal access",
    "crack software",
    "generate malware",
    "ddos attack",
    "phishing attack"
]

def is_restricted(text):
    return any(word in text for word in restricted_keywords)

In [9]:
valid_tasks_keywords = [
    "explain", "write", "generate", "create", "code",
    "image", "video", "audio", "calculate", "solve"
]

def is_valid_task(text):
    return any(word in text for word in valid_tasks_keywords)

In [10]:
def safety_layer(user_input):

    text = user_input.lower()

    # Check restricted content
    if is_restricted(text):
        return {
            "status": "blocked",
            "message": "This request is restricted. Please try a valid AI task."
        }

    # Check valid task
    if not is_valid_task(text):
        return {
            "status": "invalid",
            "message": "Unsupported request. Try tasks like text, code, image, or calculation."
        }

    return {"status": "safe"}

In [11]:
print(safety_layer("Generate malware code"))
print(safety_layer("Tell me a joke"))
print(safety_layer("Write Python code for sorting"))

{'status': 'blocked', 'message': 'This request is restricted. Please try a valid AI task.'}
{'status': 'invalid', 'message': 'Unsupported request. Try tasks like text, code, image, or calculation.'}
{'status': 'safe'}


In [12]:
def ai_system(user_input, k=3):

    # STEP 5: Safety Layer
    safety = safety_layer(user_input)

    if safety["status"] != "safe":
        return safety

    # STEP 1: Clean input
    cleaned = clean_text(user_input)

    # STEP 2: Embedding
    query_embedding = model.encode([cleaned])
    query_embedding = np.array(query_embedding).astype('float32')

    # STEP 3: FAISS search
    distances, indices = index.search(query_embedding, k)

    # STEP 4: Get best match
    best_idx = indices[0][0]
    row = df.iloc[best_idx]

    return {
        "task": row["Task Type"],
        "primary_model": row["Primary Model"],
        "fallback_model": row["Fallback Model"]
    }

In [13]:
!pip install gradio

In [14]:
import gradio as gr

def ui_system(user_input):

    result = ai_system(user_input)

    if result.get("status") == "blocked" or result.get("status") == "invalid":
        return result["message"]

    return f"""
    🧠 Task Detected: {result['task']}

    ⚡ Primary Model: {result['primary_model']}

    🚀 Fallback Model: {result['fallback_model']}
    """

In [16]:
interface = gr.Interface(
    fn=ui_system,
    inputs=gr.Textbox(lines=3, placeholder="Enter your prompt here..."),
    outputs="text",
    title="🤖 AI Model Orchestration",
    description="Enter a prompt and get the best AI model suggestion"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e36b14ea1e919e6d1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
with gr.Blocks() as demo:

    gr.Markdown("# 🤖 AI Model Router")
    gr.Markdown("Enter a prompt and get best model suggestion")

    input_box = gr.Textbox(label="Your Prompt", lines=4)

    output_box = gr.Textbox(label="Result", lines=20)  # 🔥 increased height

    btn = gr.Button("Suggest Model")

    btn.click(fn=ui_system, inputs=input_box, outputs=output_box)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://796e74c9f7546055b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
Edit → Clear all outputs

SyntaxError: invalid character '→' (U+2192) (2105537903.py, line 1)